# Market depth

The same notebook as [`../market_depth.ipynb`](../market_depth.ipynb), written against
[`ib_async`](https://github.com/ib-api-reloaded/ib_async) instead of the TWS API
shape. The library is unmodified and installed as usual; `ibx.ib_async.attach`
replaces the one layer of it that expects a socket to a gateway.

The book, and which venues will answer for it.

## Connecting

`IB.connect` was written for a gateway, so it takes a host, a port and a client
id. Here it takes none of them: the credentials go to `attach`, and there is no
local process to reach.

`ib.sleep()` rather than `time.sleep()` throughout. The library's loop runs on
this thread, and a plain sleep stops it — every stream then reads as dead.

In [ ]:
import os
from dotenv import load_dotenv
from ib_async import IB, util
import ibx.ib_async

util.startLoop()
load_dotenv()

ib = ibx.ib_async.attach(
    IB(),
    username=os.environ["IB_USERNAME"],
    password=os.environ["IB_PASSWORD"],
    paper=True,
)
ib.connect()          # names no host: there is no gateway to name

print(f"connected: {ib.isConnected()}")
print(f"accounts:  {ib.managedAccounts()}")

## Which venues answer

Depth is entitled per venue. This is what this account may ask.

In [ ]:
venues = ib.reqMktDepthExchanges()
print(f"{len(venues)} entries\n")
for v in venues[:12]:
    print(f"{v.exchange:12} {v.secType:6} {v.serviceDataType}")

## The book

A book asked for at a named venue is answered by that venue. Asked for at
none, it is acknowledged and may produce nothing, which is what an account
with no aggregate entitlement is told.

In [ ]:
from ib_async import Stock

spy = Stock("SPY", "SMART", "USD")
ib.qualifyContracts(spy)

ticker = ib.reqMktDepth(spy, numRows=10, isSmartDepth=True)
ib.sleep(5)

print(f"{'bid':>22}     {'ask':<22}")
for i in range(min(10, max(len(ticker.domBids), len(ticker.domAsks)))):
    b = ticker.domBids[i] if i < len(ticker.domBids) else None
    a = ticker.domAsks[i] if i < len(ticker.domAsks) else None
    lhs = f"{b.size:>8} @ {b.price:<8.2f} {b.marketMaker:<4}" if b else " " * 22
    rhs = f"{a.marketMaker:<4} {a.price:>8.2f} @ {a.size:<8}" if a else ""
    print(f"{lhs}     {rhs}")

## Watching it move

In [ ]:
def on_depth(t):
    top_bid = t.domBids[0].price if t.domBids else None
    top_ask = t.domAsks[0].price if t.domAsks else None
    print(f"{top_bid}  /  {top_ask}")

ticker.updateEvent += on_depth
ib.sleep(10)
ticker.updateEvent -= on_depth

In [ ]:
ib.cancelMktDepth(spy, isSmartDepth=True)
ib.disconnect()